# Fine-tuning — full protocol (stage 2a)

This notebook implements the **full VisDrone fine-tune** described in
[docs/proposal.md](../docs/proposal.md) §"Models and training strategy". The
baseline notebook ([notebooks/baseline.ipynb](../notebooks/baseline.ipynb)) ran a
deliberately short 5-epoch stage-1 fine-tune to get the pipeline working; this
notebook runs the **real** training that produces the checkpoints used for all
reported results.

Exactly **two models** are trained from COCO-pretrained weights — `yolo26m` and
`yolo26x` — with **one shared configuration**, so the only difference between the
two runs is model size and its pretrained weights. Training always uses
`imgsz=640`; the resulting checkpoints are reused unchanged at 640 / 1280 /
tiling inference (no 1280 training stage, by construction of the study).

Protocol (fixed before any result is examined):

* all layers trainable from epoch 1 (no freezing, no staged LR);
* default YOLO optimizer, `lr0≈0.01` scaled to the effective batch
  (`nbs=64`), `warmup_epochs=3`, cosine LR (`cos_lr=True`), `weight_decay=5e-4`;
* `max_epochs=50`, early stopping `patience=15` on validation mAP50–95, best
  checkpoint kept;
* Ultralytics default augmentation, `close_mosaic=10`;
* seed 42;
* effective batch size fixed once by a short pilot and recorded with every run.

> The full run takes hours on a T4/consumer GPU (~2.5 h for `yolo26m`, ~5 h for
> `yolo26x` on an 8 GB laptop in the stage-1 pilot). Run it on a Kaggle GPU
> session; see [docs/finetuning.md](../docs/finetuning.md).


## 1. Setup

Fixed protocol constants, paths and device. On Kaggle, point `VISDRONE_DATA` at
the prepared dataset directory (the one containing `visdrone.yaml` and the
`images/`, `labels/`, `manifests/` folders).


In [ ]:
from __future__ import annotations

import json
import os
import sys
import time
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import pandas as pd
import torch
from ultralytics import YOLO

# ---- fixed training protocol (proposal) ----------------------------------- #
SEED = 42
IMG_SIZE = 640  # both models trained at 640; inference may use 1280/tiling
MAX_EPOCHS = 50  # max_epochs=50
PATIENCE = 15  # early stopping on val mAP50-95
CLOSE_MOSAIC = 10  # last 10 epochs on clean geometry
WARMUP_EPOCHS = 3
COS_LR = True
LR0 = 0.01  # scaled to the effective batch (norm_batch=64)
NORM_BATCH = 64  # 'nbs': nominal batch used for LR scaling
WEIGHT_DECAY = 5e-4
OPTIMIZER = "auto"  # default YOLO optimizer
FREEZE = None  # all layers trainable from epoch 1
MODEL_NAMES = ["yolo26m", "yolo26x"]

# ---- pilot (fix the effective batch size) ----------------------------------#
RUN_PILOT = True
PILOT_MODEL = "yolo26x"  # memory-heaviest model decides the batch
PILOT_CANDIDATES = [4, 8, 16]
PILOT_FRACTION = 0.02
PILOT_EPOCHS = 1
BATCH_OVERRIDE: int | None = None  # set to skip the pilot and pin the batch

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# data/ may sit next to the repo or be mounted (Kaggle); override with VISDRONE_DATA.
DATA_DIR = Path(os.environ.get("VISDRONE_DATA", ROOT / "data"))
TRAIN_YAML = DATA_DIR / "visdrone.yaml"

RUNS_DIR = ROOT / "runs"
FINETUNE_DIR = RUNS_DIR / "finetune"
PILOT_DIR = RUNS_DIR / "pilot"
for _directory in (FINETUNE_DIR, PILOT_DIR):
    _directory.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WORKERS = 8

print(f"root={ROOT}")
print(f"data={DATA_DIR}")
print(f"train yaml={TRAIN_YAML}")
print(f"device={DEVICE}")
if DEVICE == "cuda":
    print(f"gpu={torch.cuda.get_device_name(0)}")

## 2. Shared training configuration

The **same** dict feeds both training runs (and the pilot), so class names, image
size, augmentation, LR schedule and seed cannot drift between the two models. The
only keys that ever differ per run are `batch` (fixed by the pilot, identical for
both runs) and the output `project`/`name`.


In [ ]:
SHARED_CONFIG: dict[str, Any] = {
    "data": str(TRAIN_YAML),
    "imgsz": IMG_SIZE,
    "epochs": MAX_EPOCHS,
    "patience": PATIENCE,
    "optimizer": OPTIMIZER,  # default YOLO optimizer; lr0 ignored -> auto-scaled
    "lr0": LR0,  # scaled to the effective batch via nbs
    "nbs": NORM_BATCH,
    "warmup_epochs": WARMUP_EPOCHS,
    "cos_lr": COS_LR,
    "weight_decay": WEIGHT_DECAY,
    "close_mosaic": CLOSE_MOSAIC,
    "seed": SEED,
    "deterministic": True,
    "pretrained": True,
    "freeze": FREEZE,  # all layers trainable from epoch 1
    "device": DEVICE,
    "workers": WORKERS,
    "val": True,
}


def assert_data_ready() -> None:
    """Fail early if the prepared dataset is missing."""
    required = [
        TRAIN_YAML,
        DATA_DIR / "manifests" / "train.csv",
        DATA_DIR / "manifests" / "val.csv",
    ]
    missing = [str(path) for path in required if not Path(path).exists()]
    if missing:
        raise FileNotFoundError(
            "prepared VisDrone dataset not found; run notebooks/baseline.ipynb first or "
            f"set VISDRONE_DATA. Missing: {missing}"
        )


assert_data_ready()
display(pd.Series(SHARED_CONFIG, name="value").to_frame())

## 3. Pilot — fix the effective batch size

A short pilot (2 % of train, 1 epoch, no validation) is run on the **larger**
model (`yolo26x`, the memory-heaviest) for each candidate batch, stopping at the
first out-of-memory failure. The largest batch that fits becomes the **effective
batch size**, used unchanged for both fine-tuning runs and recorded in
`runs/pilot/pilot_result.json` and `runs/finetune/training_config.json`.


In [ ]:
def run_pilot(
    model_name: str = PILOT_MODEL,
    candidates: list[int] = PILOT_CANDIDATES,
    fraction: float = PILOT_FRACTION,
    epochs: int = PILOT_EPOCHS,
) -> dict[str, Any]:
    """Fix the training batch size from a short memory/throughput pilot."""
    assert_data_ready()
    rows: list[dict[str, Any]] = []
    chosen: int | None = None
    for batch in sorted(candidates):
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
        model = YOLO(f"{model_name}.pt")
        start = time.perf_counter()
        try:
            model.train(
                **{
                    **SHARED_CONFIG,
                    "batch": batch,
                    "epochs": epochs,
                    "fraction": fraction,
                    "val": False,
                    "plots": False,
                    "save": False,
                    "project": str(PILOT_DIR),
                    "name": f"pilot_{model_name}_b{batch}",
                    "exist_ok": True,
                    "verbose": False,
                }
            )
            ok, error = True, ""
        except RuntimeError as exc:  # CUDA OOM and similar
            ok, error = False, str(exc).splitlines()[0][:140]
        elapsed = time.perf_counter() - start
        peak_gb = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
        rows.append(
            {"batch": batch, "ok": ok, "peak_gpu_gb": peak_gb, "wall_s": elapsed, "error": error}
        )
        print(
            f"batch={batch}: {'ok' if ok else 'FAILED'}  "
            f"peak={peak_gb:.2f} GB  wall={elapsed:.0f}s  {error}"
        )
        if not ok:
            break
        chosen = batch
    result = {
        "model": model_name,
        "fraction": fraction,
        "candidates": rows,
        "effective_batch": chosen,
    }
    (PILOT_DIR / "pilot_result.json").write_text(json.dumps(result, indent=2))
    return result


if BATCH_OVERRIDE is not None:
    pilot = None
    effective_batch = BATCH_OVERRIDE
    print(f"pilot skipped; batch pinned to {effective_batch}")
elif RUN_PILOT:
    pilot = run_pilot()
    effective_batch = pilot["effective_batch"]
    if effective_batch is None:
        raise RuntimeError(
            "pilot found no batch that fits; lower PILOT_CANDIDATES or use a bigger GPU"
        )
else:
    pilot = None
    effective_batch = 4
    print(f"pilot disabled; using batch={effective_batch}")

print(f"effective batch size = {effective_batch} (nbs={NORM_BATCH})")

## 4. Fine-tune both models

Identical shared configuration for `yolo26m` and `yolo26x`; only model size and
pretrained weights differ. Each run keeps the validation-selected best
checkpoint (`weights/best.pt`).


In [ ]:
def fine_tune(model_name: str, batch: int) -> dict[str, Any]:
    """One full-protocol VisDrone fine-tune; returns the best checkpoint + record."""
    assert_data_ready()
    model = YOLO(f"{model_name}.pt")
    start = time.perf_counter()
    results = model.train(
        **{
            **SHARED_CONFIG,
            "batch": batch,
            "project": str(FINETUNE_DIR),
            "name": f"{model_name}_s{IMG_SIZE}",
            "exist_ok": True,
            "verbose": True,
        }
    )
    wall = time.perf_counter() - start
    save_dir = Path(results.save_dir)
    best = save_dir / "weights" / "best.pt"
    record = {
        "model": model_name,
        "weights": str(best),
        "save_dir": str(save_dir),
        "batch": batch,
        "nbs": NORM_BATCH,
        "img_size": IMG_SIZE,
        "wall_s": wall,
    }
    (save_dir / "finetune_record.json").write_text(json.dumps(record, indent=2))
    print(f"{model_name}: best={best} ({wall / 60:.1f} min, batch={batch})")
    return record


records = [fine_tune(name, effective_batch) for name in MODEL_NAMES]

training_config = {
    **SHARED_CONFIG,
    "batch": effective_batch,
    "effective_batch": effective_batch,
    "run_pilot": RUN_PILOT,
    "pilot_model": PILOT_MODEL,
    "wall_s": {record["model"]: record["wall_s"] for record in records},
    "pilot": pilot,
}
(FINETUNE_DIR / "training_config.json").write_text(
    json.dumps(training_config, indent=2, default=str)
)
display(pd.DataFrame(records))

## 5. Training summary

Validation curves and the selected epoch for each run. Only the **validation**
split is used for selection; all reported test-dev scores are computed later by
the shared evaluator.


In [ ]:
def read_results(save_dir: str | Path) -> pd.DataFrame:
    return pd.read_csv(Path(save_dir) / "results.csv")


MAP95 = "metrics/mAP50-95(B)"
MAP50 = "metrics/mAP50(B)"

summary_rows: list[dict[str, Any]] = []
for record in records:
    df = read_results(record["save_dir"])
    best = df.loc[df[MAP95].idxmax()]
    summary_rows.append(
        {
            "model": record["model"],
            "epochs_run": int(df["epoch"].max()),
            "best_epoch": int(best["epoch"]),
            "best_val_mAP50-95": float(best[MAP95]),
            "best_val_mAP50": float(best[MAP50]),
            "wall_min": record["wall_s"] / 60.0,
            "batch": record["batch"],
            "weights": record["weights"],
        }
    )
summary = pd.DataFrame(summary_rows)
summary.to_csv(FINETUNE_DIR / "finetune_summary.csv", index=False)
display(summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for record in records:
    df = read_results(record["save_dir"])
    axes[0].plot(df["epoch"], df[MAP95], label=record["model"])
    axes[1].plot(df["epoch"], df[MAP50], label=record["model"])
axes[0].set(title="validation mAP50-95", xlabel="epoch", ylabel="mAP50-95")
axes[1].set(title="validation mAP50", xlabel="epoch", ylabel="mAP50")
for ax in axes:
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.savefig(FINETUNE_DIR / "finetune_curves.png", dpi=150)
plt.show()

## 6. Checkpoints for evaluation

The best checkpoints are now ready. Downstream experiments load them with
`YOLO(path)`; to evaluate them with the tiling notebook, point its `WEIGHTS` dict
at these paths (the fine-tuned models use the identity 10-class mapping, so
`class_map=None`).


In [ ]:
checkpoints = {record["model"]: Path(record["weights"]) for record in records}
print("Fine-tuned checkpoints:")
for name, path in checkpoints.items():
    print(f"  {name}: {path} ({'found' if path.exists() else 'MISSING'})")

# Evaluation hook (see notebooks/tiling.ipynb):
# WEIGHTS = {
#     "yolo26m": RUNS_DIR / "finetune" / "yolo26m_s640" / "weights" / "best.pt",
#     "yolo26x": RUNS_DIR / "finetune" / "yolo26x_s640" / "weights" / "best.pt",
# }

## 7. Summary

* The full training protocol from the proposal is implemented with a single
  shared config for both models; the only difference is model size/weights.
* The effective batch size is fixed once by the pilot and recorded in
  `runs/pilot/pilot_result.json` and `runs/finetune/training_config.json`,
  together with the seed, LR schedule and augmentation.
* Checkpoints are selected on validation mAP50–95 (`patience=15`, max 50 epochs)
  and written to `runs/finetune/<model>_s640/weights/best.pt`.
* Weights are trained at `imgsz=640` and reused unchanged at 1280 and at tiling
  inference; scoring uses the one shared evaluator
  ([`src/evaluation/metrics.py`](../src/evaluation/metrics.py)).
* Full details and run instructions: [docs/finetuning.md](../docs/finetuning.md).
